# 07: Single-Protein Training and Evaluation

This experiment trains the compact model to reproduce one teacher structure. It is useful for checking whether the implementation can optimize, but it measures memorization of one protein rather than generalization to new proteins.

Run notebooks 01 through 06 first. Training artifacts are deliberately excluded from Git, so checkpoint-dependent cells skip with instructions when no checkpoint is available.

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, "../src")

from af2_from_scratch import AF2Config, AlphaFold2FromScratch
from af2_from_scratch.feature_extraction import msa_features, sample_batch
from af2_from_scratch.geometry import kabsch_align, kabsch_rmsd
from af2_from_scratch.losses import lddt_ca
from af2_from_scratch.teacher import load_teacher

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (9, 4)

## 1. Choose the experiment size

`AF2Config` collects model, data, and training settings. Width and depth increase parameter count; MSA rows, residue count, and recycling also affect runtime memory.

The values below match the default compact experiment, not full AlphaFold 2.

In [ ]:
cfg = AF2Config(
    c_m=64,
    c_z=64,
    c_e=32,
    c_s=128,
    n_evo=4,
    n_extra=1,
    n_ipa=2,
    n_clu=128,
    n_ext=128,
    recycles=1,
    mask_p=0.15,
    lr=1e-3,
    steps=50_000,
)
probe = AlphaFold2FromScratch(cfg)
print(
    f"this config: {sum(p.numel() for p in probe.parameters()) / 1e6:.2f}M parameters"
)
del probe

## 2. Understand the training objectives

The training script combines three losses:

```text
loss = C-alpha FAPE + 0.3 * distogram CE + 0.01 * pLDDT CE
```

- **C-alpha FAPE** compares predicted and teacher C-alpha points in every residue's local frame. Errors are clipped at 10 Å and divided by a 10 Å length scale.
- **Distogram cross-entropy** trains each residue pair to predict its teacher distance bin.
- **pLDDT cross-entropy** trains each residue to predict its realized lDDT-C-alpha. lDDT compares corresponding intramolecular distances and does not require rigid alignment.

This project distills teacher coordinates. It does not reproduce AlphaFold's complete set of structural and auxiliary losses.

## 3. Launch training

From the repository root:

```bash
python scripts/train_single.py 2>&1 | tee logs/train_single.log
```

For a long run, use a persistent terminal such as `tmux`. Edit `src/af2_from_scratch/config.py` or pass a modified `AF2Config` in the script before launching. Checkpoints are written under `checkpoints/`.

## 4. Inspect training curves

The next cell reads `logs/train_single.log`. Re-run it while training to refresh the plots.

In [ ]:
def plot_log(path="../logs/train_single.log"):
    path = Path(path)
    if not path.exists():
        print(f"no log at {path}; launch train_single.py first")
        return

    pattern = re.compile(
        r"step\s+(\d+) \| loss\s+([\d.]+) \| fape\s+([\d.]+) "
        r"\| disto\s+([\d.]+) \| conf\s+([\d.]+) "
        r"\| CA-RMSD\s+([\d.]+)"
    )
    rows = [
        match.groups()
        for line in path.read_text().splitlines()
        if (match := pattern.search(line))
    ]
    if not rows:
        print(f"no matching training rows in {path}")
        return

    steps, total, fape, distogram, confidence, rmsd = (
        torch.tensor([float(row[index]) for row in rows]) for index in range(6)
    )
    figure, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].plot(steps, total, label="total")
    axes[0].plot(steps, fape, label="FAPE")
    axes[0].plot(steps, 0.3 * distogram, label="0.3 × distogram CE")
    axes[0].set_title("training losses")
    axes[0].set_yscale("log")
    axes[0].legend()
    axes[1].plot(steps, rmsd, color="green")
    axes[1].set_title("C-alpha RMSD to teacher (Å)")
    axes[2].plot(steps, confidence, color="purple")
    axes[2].set_title("pLDDT cross-entropy")
    for axis in axes:
        axis.set_xlabel("step")
    plt.tight_layout()
    plt.show()
    print(
        f"{len(rows)} log points; last step {int(steps[-1])}; "
        f"best RMSD {rmsd.min():.2f} Å"
    )


plot_log()

## 5. Load and evaluate the latest checkpoint

Evaluation uses a fixed unmasked MSA sample and compares several recycle counts. A value of three means one initial pass plus three recycled passes.

In [ ]:
def checkpoint_order(path):
    if path.stem == "single_final":
        return float("inf")
    match = re.fullmatch(r"single_(\d+)", path.stem)
    return int(match.group(1)) if match else -1


checkpoint_paths = sorted(
    Path("../checkpoints").glob("single_*.pt"),
    key=checkpoint_order,
)
model = None
output = None
target = None
batch = None
if not checkpoint_paths:
    print("no single-protein checkpoint found; run scripts/train_single.py first")
else:
    checkpoint_path = checkpoint_paths[-1]
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    checkpoint_cfg = AF2Config(**checkpoint["cfg"])
    model = AlphaFold2FromScratch(checkpoint_cfg).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()

    features = msa_features("../examples/tautomerase/alignment.a3m")
    target = {
        name: value.to(device)
        for name, value in load_teacher(
            "../examples/tautomerase/teacher.cif",
            n_res=features["msa_aatype"].shape[1],
        ).items()
    }
    batch = {
        name: value.to(device)
        for name, value in sample_batch(
            features,
            checkpoint_cfg.n_clu,
            checkpoint_cfg.n_ext,
            mask_p=0.0,
            seed=42,
        ).items()
    }
    print("loaded", checkpoint_path.name)
    with torch.no_grad():
        for recycles in (0, 1, 3):
            output = model(batch, recycles=recycles)
            rmsd = kabsch_rmsd(output["ca"], target["CA"])
            print(f"recycles={recycles}: C-alpha RMSD = {rmsd:.2f} Å")

## 6. Compare the predicted and teacher traces

Kabsch alignment removes global rotation and translation before plotting. The per-residue errors below are aligned coordinate errors; they are different from lDDT-C-alpha, which compares intramolecular distances.

In [ ]:
if model is None:
    print("checkpoint-dependent plot skipped")
else:
    with torch.no_grad():
        output = model(batch, recycles=3)
    predicted_ca = kabsch_align(output["ca"], target["CA"]).cpu()
    teacher_ca = target["CA"].cpu()
    coordinate_errors = (predicted_ca - teacher_ca).norm(dim=-1)

    figure = plt.figure(figsize=(12, 5))
    axis = figure.add_subplot(121, projection="3d")
    axis.plot(*teacher_ca.numpy().T, "o-", ms=4, lw=1.2, label="teacher")
    axis.plot(*predicted_ca.numpy().T, "s-", ms=3, lw=1.2, label="student")
    axis.legend()
    axis.set_title(
        f"aligned C-alpha traces; RMSD {kabsch_rmsd(output['ca'], target['CA']):.2f} Å"
    )
    error_axis = figure.add_subplot(122)
    error_axis.bar(range(len(coordinate_errors)), coordinate_errors.numpy())
    error_axis.set_title("aligned coordinate error by residue")
    error_axis.set_xlabel("residue")
    error_axis.set_ylabel("error (Å)")
    plt.tight_layout()
    plt.show()

## 7. Inspect distance and confidence heads

The distogram plot uses the probability-weighted mean of representative bin centers. The first and last bins are open-ended, so this conversion is only for visualization.

Predicted pLDDT is compared directly with realized lDDT-C-alpha from the predicted and teacher distance matrices.

In [ ]:
if output is None:
    print("checkpoint-dependent head plots skipped")
else:
    boundaries = torch.linspace(2.0, 22.0, 63, device=device)
    bin_width = boundaries[1] - boundaries[0]
    distance_centers = torch.cat(
        [
            boundaries[:1] - bin_width / 2,
            (boundaries[:-1] + boundaries[1:]) / 2,
            boundaries[-1:] + bin_width / 2,
        ]
    )
    predicted_distances = output["disto_logits"].softmax(-1) @ distance_centers
    teacher_distances = torch.cdist(target["CA"], target["CA"])

    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].imshow(teacher_distances.cpu(), cmap="viridis_r")
    axes[0].set_title("teacher C-alpha distances")
    image = axes[1].imshow(predicted_distances.cpu(), cmap="viridis_r")
    axes[1].set_title("student expected distances")
    plt.colorbar(image, ax=axes, shrink=0.8, label="Å")
    plt.show()

    confidence_centers = (torch.arange(50, device=device) + 0.5) / 50
    predicted_plddt = output["plddt_logits"].softmax(-1) @ confidence_centers
    realized_lddt = lddt_ca(output["ca"], target["CA"])
    plt.plot((predicted_plddt * 100).cpu(), label="predicted pLDDT")
    plt.plot((realized_lddt * 100).cpu(), label="realized lDDT-C-alpha")
    plt.legend()
    plt.ylabel("score")
    plt.xlabel("residue")
    plt.show()

**Interpretation:** Success here shows that the implementation can fit one teacher structure. It does not show that the model learned a transferable folding rule.

Notebook 08 evaluates teacher-structure agreement on proteins excluded from multi-protein training.